***

# **BLS Queries**

***

In this file, we want to try and pull the total number of employment from BLS for different industries. To give some context, the API works by specifying a specific series id, which can be used to pull data from a specific table from BLS. It can also be used to select information from a table, parameters like specific counties or specific industry can be referenced. The survey we are trying to pull from is State and Employment, Hours, and Earnings; which can be found in the following link: https://www.bls.gov/help/hlpforma.htm#EW. 

In [ ]:
# Packages

import pandas as pd
import json
import requests

In [ ]:
# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# API key in config.py which contains: bls_key = 'key'
key = '03688c33d1194462aa72e4e97253b1e7'
key = '?registrationkey={}'.format(key)

# Series stored as a dictionary
series_dict = {
    'LNS14000003': 'White',
    'LNS14000006': 'Black',
    'LNS14000009': 'Hispanic'}

# Start year and end year
dates = ('2008', '2017')

In [ ]:
# Specify json as content type to return
headers = {'Content-type': 'application/json'}

# Submit the list of series as data
data = json.dumps({
    "seriesid": list(series_dict.keys()),
    "startyear": dates[0],
    "endyear": dates[1]})

# Post request for the data
p = requests.post(
    '{}{}'.format(url, key),
    headers=headers,
    data=data).json()['Results']['series']

In [ ]:
# Date index from first series
date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# Empty dataframe to fill with values
df = pd.DataFrame()

# Build a pandas series from the API results, p
for s in p:
    df[series_dict[s['seriesID']]] = pd.Series(
        index = pd.to_datetime(date_list),
        data = [i['value'] for i in s['data']]
        ).astype(float).iloc[::-1]

# Show last 5 results
df.tail()

In [ ]:
# Series stored as a dictionary
series_dict = {
    'LNS12000000': 'Agricultural Total Employment'} # testing a different series ID

# Start year and end year
dates = ('2022', '2024')

# Specify json as content type to return
headers = {'Content-type': 'application/json'}

# Submit the list of series as data
data = json.dumps({
    "seriesid": list(series_dict.keys()),
    "startyear": dates[0],
    "endyear": dates[1]})

# Post request for the data
p = requests.post(
    '{}{}'.format(url, key),
    headers=headers,
    data=data).json()['Results']['series']
# Date index from first series
date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# Empty dataframe to fill with values
df = pd.DataFrame()

# Build a pandas series from the API results, p
for s in p:
    df[series_dict[s['seriesID']]] = pd.Series(
        index = pd.to_datetime(date_list),
        data = [i['value'] for i in s['data']]
        ).astype(float).iloc[::-1]

# Show last 5 results
df.tail()

***

# **Running Pulls on our Counties Using MSA and Construction as Industry**

***

In [ ]:
def bls_query(series_dict, dates, api_key):

    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

    key = '?registrationkey={}'.format(api_key)

    # Specify json as content type to return
    headers = {'Content-type': 'application/json'}

    # Submit the list of series as data
    data = json.dumps({
        "seriesid": list(series_dict.keys()),
        "startyear": dates[0],
        "endyear": dates[1]})

    # Post request for the data
    p = requests.post(
        '{}{}'.format(url, key),
        headers=headers,
        data=data).json()['Results']['series']
    
    # Date index from first series
    date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

    global df

    # Empty dataframe to fill with values
    df = pd.DataFrame()

    # Build a pandas series from the API results, p
    for s in p:
        df[series_dict[s['seriesID']]] = pd.Series(
            index = pd.to_datetime(date_list),
            data = [i['value'] for i in s['data']]
            ).astype(float).iloc[::-1]

    return(df)

In [ ]:
# Making Series ID
series_list = []
front = 'SMU06'

industry = '20236000'

type = '01'

# Making a county list for multiple series id creation | In order: Sac + Placer + El Dorado Yolo, Yuba + Sutter,
county_list = ['40900', '49700']
for i in county_list:
    id = front + i + tail + industry + type
    series_list.append(id)

In [ ]:
series_list

In [ ]:
# Make the dictionary

# if do 19780 it works but not for 

# this is the test code that BLS gave
dicto = {
    'SMU19197802023800001': 'Sac, Placer, El D, Yolo'} # testing a different series ID

years = ('2000', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=dicto, dates=years, api_key=my_key)

In [ ]:
# Make the dictionary

dicto = {
    'SMU064090050010500000001': 'Sac, Placer, El D, Yolo',
    'SMU064970050012023600001': 'Yuba, Sutter'} # testing a different series ID

years = ('2000', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=dicto, dates=years, api_key=my_key)

We keep returning empty dataframes when we use our counties. If we use the example query ID 'SMU19197802023800001' which selects Specialty Trade Contractors as the industry and Des Moines-West Des Moines, IA we get a dataframe of length 27. When we ran this query with the original county value (19780) substituted for that of Sacramento county, we got length zero for our frame. It's also important to note that even on our successful pull, we only got data spanning from 2022-2024.

In [ ]:
# Using the Same survey but SIC Basis

# Example SeriesID given

# SAS0800002000011 should have data going back to 2000 - 2002

# Supersector code is contained within the industry codes

#ie 20238000 has a 20 in front, which means construction

dicto = {
    'SAS0800002000011': 'Test SeriesID'} # testing a different series ID

years = ('2000', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=dicto, dates=years, api_key=my_key)

***

## **Trying Peer MSAs**

***

In [ ]:
# Reading in Data

peers = pd.read_excel("Area Codes (1).xlsx", 'Common Groups')

peers.head(13)

In [ ]:
# Subsetting to only keep our peer counties

# Referencing intiial input as well as the excel file

peers = peers.iloc[10:33]

peers.tail()

In [ ]:
# Attempting to do a pull

# This is using the structure of the test code series id in the documentation: 'SMU19197802023800001'

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

years = ('2000', '2024')

bls_query(series_dict=dictochat, dates=years, api_key=my_key)

# UPDATE: Accidentally used CA Id for all. 

### Notes

* Unsure how great this source is, as there is not a ton of records to pull from.

* In my code, need to update it so that each series is returned separately. To further explain, take the example above. The length of values for some counties, are 27 while some are zero.

* Will also need to change the industries. I surmise that some, counties are going to have more data than other i.e big cities. 

In [ ]:
# Trying to test bullet point 3

# County is inside the list as well, so should be returning data.

dicto = {
    'SMU06418602023730001': 'San Francisco-Oakland-Berkeley, CA Metro Area'}

In [ ]:
# Going to do multiple queries using different industries

# No Results: 2023800, 20236000, 20237300

# Checking the URL link on browser (https://api.bls.gov/publicAPI/v2/timeseries/data/20237300)

# I get "invalid series for series Series 2023700" whenever I do this.
bls_query(series_dict=dicto, dates=years, api_key=my_key)

In [ ]:
# Loading state names as well 

states = pd.read_excel("state names.xlsx")

In [ ]:
# Need the codes to be 00, 01, etc
states.iloc[0:8, 0] = '0' + states.iloc[0:8, 0].astype(str)

In [ ]:
states.head()
peers.head()

In [ ]:
# make a function to extract state abbreviation  

import us

# Gets just abbrev out of the text string in peers

def extract_state(text):
    return text.split(',')[1].strip()

# Converting abbreviation to full name

def abbr_full(abbr):
    return us.states.lookup(abbr).name


# Getting each ID now using full name 

def get_id(full_state_name):
    row = states[states['state_name'] == full_state_name]
    if not row.empty:
        return row.iloc[0]['state_code']
    else:
        return None
    
peers['full_name'] = peers['Unnamed: 1'].apply(extract_state)

peers['abbr'] = peers['full_name'].str[:2]

peers['full_name'] = peers['abbr'].apply(abbr_full)

peers['State_ID'] = peers['full_name'].apply(get_id)

In [ ]:
# Changing state id to be string

peers['State_ID'] = peers['State_ID'].astype(str)

In [ ]:
# Create a function to make all of these counties into a dictionary

def dict_maker(df):

    # Formula for Series ID = Prefix + SA + State + Area + Industry + DType
    pre = "SMU"
    
    supersector = "20238000" # We can make this be chosen

    data_type = '01'

    # Making set of keys and vals for future dict
    
    keys = []

    # Have to initialize as we can't use 0 in for loop

    # defo a better way

    first_key = pre + '48' + df.iloc[0, 0] + supersector + data_type

    keys.append(first_key)

    # Now empty list for values in the future dict

    vals = []

    first_val = df.iloc[0,1]
    
    vals.append(first_val)

    for i in range(len(df)):

        # Getting each code
        area_code = df.iloc[i, 0]

        state = peers.iloc[i, 3]

        # Making each SeriesID
        series_id = pre + state + area_code + supersector + data_type

        # Adding to the keylist for future dictionary
        keys.append(series_id)

        val = df.iloc[i, 1]

        vals.append(val)

    result = {k: v for k, v in zip(keys, vals)}

    return result
        
dictochat = dict_maker(df = peers)

In [ ]:
dictochat

In [ ]:
# Running with fixed code

bls_query(series_dict=dictochat, dates=years, api_key=my_key)

Same issue as before, but this should be an easy fix. Again, 27 is the max length we've observed. Farthest data I've seen this survey go back is 2014, is this enough?

In [ ]:
#SMU06409009000000001

sac_dict = {
    'SMU06409009000000001': 'Sac, Placer, El D, Yolo'} # testing a different series ID

years = ('2014', '2024')

# Call the func

my_key = '03688c33d1194462aa72e4e97253b1e7'

bls_query(series_dict=sac_dict, dates=years, api_key=my_key)

In [ ]:
# Let's update bls_query() so that we can make it so that each series is uniform and has 120 rows for all

def bls_query_update(series_dict, dates, api_key):

    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

    key = '?registrationkey={}'.format(api_key)

    # Specify json as content type to return
    headers = {'Content-type': 'application/json'}

    # Submit the list of series as data
    data = json.dumps({
        "seriesid": list(series_dict.keys()),
        "startyear": dates[0],
        "endyear": dates[1]})

    # Post request for the data
    p = requests.post(
        '{}{}'.format(url, key),
        headers=headers,
        data=data).json()['Results']['series']
    
    # Date index from first series
    date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

    global df

    # Empty dataframe to fill with values
    df = pd.DataFrame()

    # Build a pandas series from the API results, p

    # We want to make all industries consistent, meaning all counties have 120 rows
    for s in p:
        county_name = series_dict[s['seriesID']]
        county_data = {f"{i['year']}-{i['period'][1:]}-01": float(i['value']) if 'value' in i else 0 for i in s['data']}

        df[county_name] = pd.Series(county_data)

    df.index = pd.to_datetime(date_list)

    return(df)

bls_query_update(series_dict=dictochat, dates = (2000, 2024), api_key = my_key)

***

## **Pulling All Subsectors With All Peers**

***

In [ ]:
peers

In [ ]:
# Create a function to make all of these counties into a dictionary

def dict_maker(df, sector):

    # Formula for Series ID = Prefix + SA + State + Area + Industry + DType
    pre = "SMU"
    
    data_type = '01'

    # Making set of keys and vals for future dict
    
    keys = []

    # Have to initialize as we can't use 0 in for loop

    # defo a better way

    first_key = pre + '48' + df.iloc[0, 0] + df.iloc[0,3] + data_type

    keys.append(first_key)

    # Now empty list for values in the future dict

    vals = []

    first_val = df.iloc[0,1]
    
    vals.append(first_val)

    for i in range(len(df)):

        # Getting each code
        area_code = df.iloc[i, 0]

        state = peers.iloc[i, 3]

        # Making each SeriesID
        series_id = pre + state + area_code + sector + data_type

        # Adding to the keylist for future dictionary
        keys.append(series_id)

        val = df.iloc[i, 1]

        vals.append(val)

    result = {k: v for k, v in zip(keys, vals)}

    return result
        
dictochat = dict_maker(df = peers)

In [ ]:
# Need to update bls_query() so that it doesn't only do the first 10 years

# Let's update bls_query() so that we can make it so that each series is uniform and has 120 rows for all

def bls_query_update(series_dict, dates, api_key):

    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

    key = '?registrationkey={}'.format(api_key)

    start_year = dates[0]

    end_year = dates[1]

    # Specify json as content type to return
    headers = {'Content-type': 'application/json'}

    # Initialize empty df for assignment later

    df = pd.DataFrame()

    # Determine the number of years to query at once
    year_step = 10

    # Loop through the specified range of years in step intervals
    for year_range_start in range(start_year, end_year + 1, year_step):
        year_range_end = min(year_range_start + year_step - 1, end_year)

        # Specify the date range for the current iteration
        dates = (year_range_start, year_range_end)

        # Submit the request for the current date range
        data = json.dumps({
            "seriesid": list(series_dict.keys()),
            "startyear": dates[0],
            "endyear": dates[1]})
        response = requests.post('{}{}'.format(url, key), headers=headers, data=data).json()

        # Extract data from the response and concatenate to the DataFrame
        if 'Results' in response and 'series' in response['Results']:
            p = response['Results']['series']
            date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]
            temp_df = pd.DataFrame(index=pd.to_datetime(date_list))

            for s in p:
                county_name = series_dict[s['seriesID']]
                county_data = {f"{i['year']}-{i['period'][1:]}-01": float(i['value']) if 'value' in i else None for i in s['data']}
                temp_df[county_name] = pd.Series(county_data)

            df = pd.concat([df, temp_df], axis=0)

    return df


# Now, it should return a single dataframe for a single subsector.

# This frame should contain all of the counties, with dates from 2000-2024.

# Additionally, if a county doesn't have data for a specific month, that cell should be empty

bls_query_update(series_dict=dictochat, dates = (2000, 2024), api_key = my_key)

In [ ]:
# Confirming it gets even 2023 data using SMU06409000000000001

sac = bls_query_update(series_dict=sac_dict, dates = (2000, 2024), api_key = my_key)

In [ ]:
sac

In [ ]:
# Now let's run the final function

# Should also try and make it so that you can choose what counties you want.

# This will be done when we make the function in tandem with excel

def full_bls(sector_list, df, key):
    
    # Initialize an empty list so we can iterate over multiple dictionaries
    sector_chamber = []

    # Initialize empty list for each dataframe we will end up making
    df_chamber = []

    # Loop over each sector we want to test
    for i in sector_list:
        sector_chamber.append(dict_maker(df, i))

    # Now with the sector_holders list containing each set of series we want, we can run our query function iteratively
    
    # Iteratively make each dataframe
    for sector_dict in sector_chamber:
        df_chamber.append(bls_query_update(sector_dict, dates = (2000, 2024), api_key = key))

    return(df_chamber)
# Perhaps we can multiprocess this. For loops very bad for efficiency, and this is going to be a p lengthy process. 


In [ ]:
import multiprocessing
import time

# This is a multiprocess function for improved efficiency.
def full_bls(sector_list, df, key):

    # Empty list to store results
    results = []

    # Define a function to process each sector, use time for rate limit
    def process_sector(sector):
        time.sleep(10)
        return bls_query_update(dict_maker(df, sector), dates=(2000, 2024), api_key=key)

    # Create a pool of worker processes
    with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
        
        # Map the sectors to worker processes
        results = pool.map(process_sector, sector_list)

    return results

The API returns up to 10 years of data for up to 25 time series. We will have to separate the queries in half, and then stack the frames on top of each other. 

Also, we can actually pull all the way back to 1990. So maybe we have to do this process three times. 

In [ ]:
# Testing Agg = Total Nonfarm - Total Priv

sectors = ['00000000', '05000000', '08000000']
dfs = full_bls(sectors, peers, my_key)

In [ ]:
# Trying to find agriculture
trade = ['41000000']
trade_df = full_bls(trade, peers, my_key)

In [ ]:
# This most recent line i ran I think i ran out of queries. 

trade_df[0]

In [ ]:
# Let's test this first function

sectors = ['00000000', '10000000', '20000000']
dfs = full_bls(sectors, peers, my_key)

In [ ]:
print(dfs[0].shape)
print(dfs[1].shape)
print(dfs[2].shape)

In [ ]:
# Testing rows 335 and 339 for Garrett

# Trying to see if these dfs are different given our peer counties
gov_ed = ['90931611', '90936111']
gov_ed_dfs = full_bls(gov_ed, peers, my_key)

In [ ]:
# The test one had data from 2000, 2022

# Diagnosing if my code is correctly looping all the years

gov_ed = ['90931611', '90936111']
gov_ed_dfs = full_bls(gov_ed, peers, my_key)

In [ ]:
display(gov_ed_dfs[0])

In [ ]:
display(gov_ed_dfs[1])

In [ ]:
# Let's test this first function

sectors = ['00000000', '10000000', '20000000']
dfs = full_bls(sectors, peers, my_key)

In [ ]:
dfs[0]

In [ ]:
my_key